# Colab 04 — AML approved-drug screening

## Goal

Stage AML metadata, selected standard-v2 checkpoints, and the AML residue cache plus provenance sidecar; run the provenance-safe dual-model screen, rank candidates, and sync outputs to Drive.

Outputs are computationally prioritised drug-repurposing candidates—not experimentally or clinically validated treatments. **Screening and sync default to off.**


## Setup


In [ ]:
from pathlib import Path
from importlib import import_module
import json
import os
import shutil
import subprocess
import sys

GIT_URL = "https://github.com/Thebluesky303/msc-dissertation-drug-repurposing.git"
BRANCH = "v2-refactor"
REPO_DIR = Path("/content/msc-dissertation-drug-repurposing")
WORKSPACE_DIR = Path("/content/dti_workspace")
DRIVE_ROOT = Path("/content/drive/MyDrive/msc-dissertation-drug-repurposing")


def configure_workspace_environment() -> None:
    # Configure all paths before importing config or any src module.
    os.environ["AML_GNN_PROJECT_ROOT"] = str(REPO_DIR)
    os.environ["AML_GNN_DATA_DIR"] = str(WORKSPACE_DIR / "data")
    os.environ["AML_GNN_MODELS_DIR"] = str(WORKSPACE_DIR / "checkpoints")
    os.environ["AML_GNN_RESULTS_DIR"] = str(WORKSPACE_DIR / "results")
    if str(REPO_DIR) not in sys.path:
        sys.path.insert(0, str(REPO_DIR))


def require_api(module_name: str, attribute: str):
    module = import_module(module_name)
    if not hasattr(module, attribute):
        raise RuntimeError(f"Missing required v2 API: {module_name}.{attribute}")
    return getattr(module, attribute)


def copy_directory(source: Path, destination: Path) -> None:
    if not source.exists():
        raise FileNotFoundError(source)
    destination.parent.mkdir(parents=True, exist_ok=True)
    shutil.copytree(source, destination, dirs_exist_ok=True)


print(json.dumps({
    "git_url": GIT_URL,
    "branch": BRANCH,
    "repo": str(REPO_DIR),
    "workspace": str(WORKSPACE_DIR),
    "drive_root": str(DRIVE_ROOT),
}, indent=2))


In [ ]:
BASELINE_CHECKPOINT_NAME = "selected_baseline.pt"
ADVANCED_CHECKPOINT_NAME = "selected_advanced.pt"
ESM2_MODEL_NAME = "facebook/esm2_t6_8M_UR50D"
ESM2_MODEL_REVISION = None  # pin an immutable Hugging Face commit for final runs
AML_CACHE_NAME = "aml_targets_esm2_t6_8m_residues.pt"
AML_CACHE_PATH = WORKSPACE_DIR / "caches" / "esm2" / AML_CACHE_NAME
AML_CACHE_METADATA_PATH = AML_CACHE_PATH.with_suffix(".metadata.json")

RUN_STAGE_INPUTS = False
RUN_AML_EMBEDDING = False
RUN_SCREENING = False
RUN_SYNC_TO_DRIVE = False


## Steps


In [ ]:
if RUN_STAGE_INPUTS:
    copy_directory(DRIVE_ROOT / "data" / "aml", WORKSPACE_DIR / "data" / "aml")
    checkpoint_source = DRIVE_ROOT / "checkpoints"
    copy_directory(checkpoint_source, WORKSPACE_DIR / "checkpoints")
    cache_source_directory = DRIVE_ROOT / "caches" / "esm2"
    cache_destination_directory = WORKSPACE_DIR / "caches" / "esm2"
    cache_destination_directory.mkdir(parents=True, exist_ok=True)
    cache_artifacts = (
        cache_source_directory / AML_CACHE_PATH.name,
        cache_source_directory / AML_CACHE_METADATA_PATH.name,
    )
    present_cache_artifacts = [path.exists() for path in cache_artifacts]
    if any(present_cache_artifacts) and not all(present_cache_artifacts):
        raise FileNotFoundError(
            "Drive must contain both the AML cache tensor and metadata sidecar."
        )
    if all(present_cache_artifacts):
        for cache_artifact in cache_artifacts:
            shutil.copy2(
                cache_artifact,
                cache_destination_directory / cache_artifact.name,
            )
    elif not RUN_AML_EMBEDDING:
        raise FileNotFoundError(
            "No complete AML cache exists in Drive. Either stage both .pt and "
            ".metadata.json files or explicitly enable RUN_AML_EMBEDDING."
        )
else:
    print("AML inputs/checkpoints/cache staging skipped: RUN_STAGE_INPUTS is False.")


In [ ]:
if RUN_AML_EMBEDDING:
    configure_workspace_environment()
    import pandas as pd

    from src.preprocessing import build_esm2_residue_embedding_cache

    target_path = WORKSPACE_DIR / "data" / "aml" / "aml_targets.csv"
    if not target_path.exists():
        raise FileNotFoundError(
            f"Stage the established AML target table first: {target_path}"
        )
    targets_for_cache = pd.read_csv(target_path)
    residue_embedding_cache = build_esm2_residue_embedding_cache(
        dataframe=targets_for_cache,
        output_path=AML_CACHE_PATH,
        model_name=ESM2_MODEL_NAME,
        model_revision=ESM2_MODEL_REVISION,
    )
    if not AML_CACHE_METADATA_PATH.exists():
        raise FileNotFoundError(
            "Cache builder did not create the required metadata sidecar: "
            f"{AML_CACHE_METADATA_PATH}"
        )
    print({
        "cached_targets": len(residue_embedding_cache),
        "cache_path": str(AML_CACHE_PATH),
        "metadata_path": str(AML_CACHE_METADATA_PATH),
    })
else:
    print("AML ESM-2 cache generation skipped: RUN_AML_EMBEDDING is False.")


In [ ]:
screening_completed = False
if RUN_SCREENING:
    configure_workspace_environment()
    import hashlib
    import pandas as pd

    from src.preprocessing import load_residue_embedding_cache
    from src.screening import (
        build_screening_pairs,
        prepare_aml_targets,
        prepare_approved_drugs,
        run_dual_model_aml_screening,
    )

    aml_dir = WORKSPACE_DIR / "data" / "aml"
    output_dir = WORKSPACE_DIR / "results" / "screening"
    output_dir.mkdir(parents=True, exist_ok=True)
    raw_drugs = pd.read_csv(aml_dir / "approved_drugs_raw.csv")
    raw_targets = pd.read_csv(aml_dir / "aml_targets.csv")
    drugs, rejected = prepare_approved_drugs(raw_drugs, return_rejections=True)
    targets = prepare_aml_targets(raw_targets)
    pairs = build_screening_pairs(drugs, targets)
    residue_embedding_cache = load_residue_embedding_cache(
        AML_CACHE_PATH
    )
    result = run_dual_model_aml_screening(
        pairs=pairs,
        baseline_checkpoint=(
            WORKSPACE_DIR / "checkpoints" / BASELINE_CHECKPOINT_NAME
        ),
        advanced_checkpoint=(
            WORKSPACE_DIR / "checkpoints" / ADVANCED_CHECKPOINT_NAME
        ),
        residue_embedding_cache=residue_embedding_cache,
        batch_size=32,
    )

    def fingerprint_file(path, chunk_size=1024 * 1024):
        path = Path(path)
        if not path.is_file():
            raise FileNotFoundError(path)
        digest = hashlib.sha256()
        with path.open("rb") as file:
            for chunk in iter(lambda: file.read(chunk_size), b""):
                digest.update(chunk)
        return {
            "path": str(path.resolve()),
            "bytes": path.stat().st_size,
            "sha256": digest.hexdigest(),
        }

    screening_provenance = {
        "runner_provenance": result.provenance,
        "input_files": {
            "approved_drugs_raw_csv": fingerprint_file(
                aml_dir / "approved_drugs_raw.csv"
            ),
            "aml_targets_csv": fingerprint_file(
                aml_dir / "aml_targets.csv"
            ),
            "residue_cache_tensor": fingerprint_file(AML_CACHE_PATH),
            "residue_cache_metadata": fingerprint_file(
                AML_CACHE_METADATA_PATH
            ),
        },
    }

    drugs.to_csv(aml_dir / "approved_drugs_metadata.csv", index=False)
    rejected.to_csv(output_dir / "approved_drug_rejections.csv", index=False)
    pairs.to_csv(output_dir / "aml_approved_drug_pairs.csv", index=False)
    result.baseline_predictions.to_csv(
        output_dir / "baseline_predictions_full.csv", index=False
    )
    result.advanced_predictions.to_csv(
        output_dir / "advanced_predictions_full.csv", index=False
    )
    for model_name, ranking in (
        ("baseline", result.baseline_rankings),
        ("advanced", result.advanced_rankings),
    ):
        ranking.top_10_overall.to_csv(
            output_dir / f"{model_name}_top10.csv", index=False
        )
        ranking.top_20_overall.to_csv(
            output_dir / f"{model_name}_top20.csv", index=False
        )
        ranking.top_5_per_target.to_csv(
            output_dir / f"{model_name}_top5_per_target.csv", index=False
        )
    result.comparison.pair_comparison.to_csv(
        output_dir / "pair_rank_comparison.csv", index=False
    )
    result.comparison.shared_top_drugs.to_csv(
        output_dir / "shared_top_drugs.csv", index=False
    )
    result.comparison.top_pair_overlap.to_csv(
        output_dir / "top_pair_overlap.csv", index=False
    )
    result.comparison.target_agreement.to_csv(
        output_dir / "target_agreement.csv", index=False
    )
    (output_dir / "comparison_summary.json").write_text(
        json.dumps(
            result.comparison.summary,
            indent=2,
            sort_keys=True,
            default=str,
        ),
        encoding="utf-8",
    )
    (output_dir / "screening_provenance.json").write_text(
        json.dumps(
            screening_provenance,
            indent=2,
            sort_keys=True,
            default=str,
        ),
        encoding="utf-8",
    )
    screening_completed = True
    print(json.dumps(screening_provenance, indent=2, default=str))
else:
    print("AML screening skipped: RUN_SCREENING is False.")


In [ ]:
if RUN_SYNC_TO_DRIVE:
    if not screening_completed:
        raise RuntimeError("Run and inspect screening before syncing.")
    copy_directory(
        WORKSPACE_DIR / "results" / "screening",
        DRIVE_ROOT / "results" / "screening",
    )
    cache_destination = DRIVE_ROOT / "caches" / "esm2"
    cache_destination.mkdir(parents=True, exist_ok=True)
    for cache_artifact in (AML_CACHE_PATH, AML_CACHE_METADATA_PATH):
        if not cache_artifact.exists():
            raise FileNotFoundError(
                f"Refusing to sync without AML cache artifact: {cache_artifact}"
            )
        shutil.copy2(
            cache_artifact,
            cache_destination / cache_artifact.name,
        )
    print("Screening outputs synced to Drive.")
else:
    print("Drive sync skipped: RUN_SYNC_TO_DRIVE is False.")


## Checks

Confirm both AML cache artifacts were staged or deliberately generated with a pinned immutable `ESM2_MODEL_REVISION`, drug deduplication/rejections, pair counts, finite scores, compatible v2 checkpoint provenance, dataset-specific score labels, top-10/top-20/top-five-per-target exports, rank differences, target-specific agreement, and provenance JSON before sync. The provenance JSON must contain chunked SHA-256 and byte size for the raw approved-drug CSV, AML targets CSV, cache tensor, and cache sidecar.

## Next steps

Report hypotheses with appropriate uncertainty and limitations. Do not describe model rankings as therapeutic efficacy or clinical validation.
